# Temporal Difference Learning
## TD(0), SARSA, and Q-Learning: Bootstrapping Methods for RL

This notebook provides a self-contained, from-scratch implementation of **Temporal Difference (TD)** learning methods for reinforcement learning. We build custom environments and implement TD(0) prediction, SARSA (on-policy control), and Q-Learning (off-policy control). Every concept is developed from first principles with detailed derivations and visualizations.

**What you'll learn:**
1. TD(0) prediction: bootstrapping value estimates without waiting for episode completion
2. SARSA: on-policy TD control that learns while following an epsilon-greedy policy
3. Q-Learning: off-policy TD control that learns the optimal policy regardless of behavior
4. Key differences between Monte Carlo, TD, and Dynamic Programming methods
5. CliffWalking comparison: SARSA (safe path) vs Q-Learning (optimal path)

**Prerequisites:** MC methods (Notebook 2), Bellman equations (Notebook 1).

**References:**
- Sutton & Barto, *Reinforcement Learning: An Introduction*, 2nd Ed., MIT Press, 2018, Chapter 6.
- Watkins, C.J.C.H., *Learning from Delayed Rewards*, PhD Thesis, King's College, Cambridge, 1989.
- Rummery & Niranjan, *On-line Q-Learning Using Connectionist Systems*, Technical Report, Cambridge, 1994.

---
## 1. Imports and Configuration

In [ ]:
# ============================================================
#  Imports and Configuration
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
from typing import Tuple, List, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# ---- reproducibility ----
SEED = 42
rng = np.random.default_rng(SEED)

# ---- plot style ----
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

# ---- color palette ----
C_PRIMARY = 'steelblue'
C_SECONDARY = 'coral'
C_TERTIARY = 'seagreen'
C_ACCENT = 'goldenrod'
C_HIGHLIGHT = 'mediumpurple'

# ---- algorithm constants ----
ALPHA = 0.1       # learning rate
GAMMA = 1.0       # discount factor (undiscounted for grid tasks)
EPSILON = 0.1     # exploration rate
N_EPISODES = 500  # training episodes

print("Configuration complete.")
print(f"  SEED={SEED}, ALPHA={ALPHA}, GAMMA={GAMMA}, EPSILON={EPSILON}, N_EPISODES={N_EPISODES}")

---
## 2. From Monte Carlo to Temporal Difference

### The Key Insight

**Monte Carlo (MC)** methods must wait until the end of an episode to update value estimates, using the actual return $G_t$:

$$V(S_t) \leftarrow V(S_t) + \alpha \left[ G_t - V(S_t) \right]$$

**Temporal Difference (TD)** methods, by contrast, **bootstrap** — they update estimates using other estimates. TD(0) updates after every single step, using the observed reward $R_{t+1}$ plus the discounted estimate of the next state's value:

$$\boxed{V(S_t) \leftarrow V(S_t) + \alpha \left[ R_{t+1} + \gamma V(S_{t+1}) - V(S_t) \right]}$$

The quantity inside the brackets is the **TD error**:

$$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$$

This is the difference between the **TD target** $R_{t+1} + \gamma V(S_{t+1})$ and the current estimate $V(S_t)$.

### Bias-Variance Tradeoff

| Property | MC | TD(0) |
|---|---|---|
| **Target** | $G_t$ (actual return) | $R_{t+1} + \gamma V(S_{t+1})$ (bootstrapped) |
| **Bias** | Unbiased | Biased (uses estimate $V(S_{t+1})$) |
| **Variance** | High (depends on full trajectory) | Low (depends on single transition) |
| **Online** | No (must wait for episode end) | Yes (update every step) |
| **Convergence** | Converges to $V^\pi$ | Converges to $V^\pi$ (with suitable $\alpha$) |

---
## 3. SARSA: On-Policy TD Control

SARSA extends TD(0) from **prediction** (estimating $V$) to **control** (estimating $Q$ and improving the policy). The name comes from the quintuple $(S_t, A_t, R_{t+1}, S_{t+1}, A_{t+1})$.

$$\boxed{Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t) \right]}$$

SARSA is **on-policy**: it evaluates and improves the same $\epsilon$-greedy policy it uses to select actions. Because it accounts for the exploratory actions it will actually take, SARSA tends to find **safe** policies that avoid dangerous states near which exploration could be costly.

---
## 4. Q-Learning: Off-Policy TD Control

Q-Learning separates the **behavior policy** (which selects actions, typically $\epsilon$-greedy) from the **target policy** (which is being learned, the greedy policy). The update uses the **maximum** Q-value over all actions in the next state:

$$\boxed{Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma \max_{a'} Q(S_{t+1}, a') - Q(S_t, A_t) \right]}$$

Q-Learning is **off-policy**: the learned Q-values converge to $Q^*$ (the optimal action-value function) regardless of the exploration strategy. This means Q-Learning finds the **optimal** path, even if that path is risky during training.

---
## 5. Comparison: MC vs TD vs DP

| Dimension | Monte Carlo | TD Learning | Dynamic Programming |
|---|---|---|---|
| **Bootstraps?** | No (uses actual returns) | Yes (uses estimated values) | Yes (uses estimated values) |
| **Samples?** | Yes (learns from experience) | Yes (learns from experience) | No (uses full model) |
| **Model-free?** | Yes | Yes | No (requires transition model) |
| **Online?** | No (episodic only) | Yes (updates every step) | N/A (planning method) |
| **Handles continuing tasks?** | No | Yes | Yes |
| **Variance** | High | Low | Zero (deterministic) |
| **Bias** | None | Some (bootstrapping) | None (exact model) |

---
## 6. GridWorld Environment (4x4)

In [ ]:
# ============================================================
#  GridWorld Environment (4x4)
# ============================================================

class GridWorld:
    """
    A simple 4x4 deterministic grid world for TD(0) prediction.
    
    Terminal state at (3,3) with reward 0. All other transitions
    give reward -1. Agent cannot move outside the grid.
    
    Actions: 0=UP, 1=RIGHT, 2=DOWN, 3=LEFT
    States are represented as (row, col) tuples.
    """
    
    # ---- action definitions ----
    UP, RIGHT, DOWN, LEFT = 0, 1, 2, 3
    ACTION_NAMES = ['UP', 'RIGHT', 'DOWN', 'LEFT']
    ACTION_DELTAS = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    
    def __init__(self, rows: int = 4, cols: int = 4):
        """
        Args:
            rows: number of rows in the grid
            cols: number of columns in the grid
        """
        self.rows = rows
        self.cols = cols
        self.n_states = rows * cols
        self.n_actions = 4
        self.terminal_state = (rows - 1, cols - 1)
        self.state = None
    
    def reset(self) -> Tuple[int, int]:
        """Reset to top-left corner.
        
        Returns:
            state: (row, col) starting position
        """
        self.state = (0, 0)
        return self.state
    
    def step(self, action: int) -> Tuple[Tuple[int, int], float, bool]:
        """Take an action in the environment.
        
        Args:
            action: integer action (0=UP, 1=RIGHT, 2=DOWN, 3=LEFT)
        
        Returns:
            next_state: (row, col) after transition
            reward: -1 for all transitions (0 when already at terminal)
            done: True if next_state is terminal
        """
        if self.state == self.terminal_state:
            return self.state, 0.0, True
        
        dr, dc = self.ACTION_DELTAS[action]
        new_r = max(0, min(self.rows - 1, self.state[0] + dr))
        new_c = max(0, min(self.cols - 1, self.state[1] + dc))
        self.state = (new_r, new_c)
        
        done = (self.state == self.terminal_state)
        return self.state, -1.0, done
    
    def is_terminal(self, state: Tuple[int, int]) -> bool:
        """Check if a state is terminal."""
        return state == self.terminal_state
    
    def all_states(self) -> List[Tuple[int, int]]:
        """Return list of all states as (row, col) tuples."""
        return [(r, c) for r in range(self.rows) for c in range(self.cols)]


# Quick test
gw = GridWorld()
s = gw.reset()
print(f"Start: {s}")
s, r, done = gw.step(GridWorld.RIGHT)
print(f"After RIGHT: state={s}, reward={r}, done={done}")
s, r, done = gw.step(GridWorld.DOWN)
print(f"After DOWN:  state={s}, reward={r}, done={done}")
print(f"Terminal state: {gw.terminal_state}")

---
## 7. TD(0) Prediction Algorithm

In [ ]:
# ============================================================
#  TD(0) Prediction
# ============================================================

def td0_prediction(
    env: GridWorld,
    policy: Dict[Tuple[int, int], np.ndarray],
    n_episodes: int,
    alpha: float,
    gamma: float,
    seed: int = SEED
) -> Tuple[Dict, List[float], List[List[float]]]:
    """
    TD(0) prediction: estimate V^pi using temporal difference updates.
    
    Args:
        env: GridWorld environment instance
        policy: mapping from state -> action probability array, shape (n_actions,)
        n_episodes: number of episodes to run
        alpha: learning rate (step size)
        gamma: discount factor
        seed: random seed for reproducibility
    
    Returns:
        V: dict mapping state -> estimated value
        episode_rewards: list of total rewards per episode, shape (n_episodes,)
        td_errors_history: list of TD errors per episode, shape (n_episodes, variable)
    """
    local_rng = np.random.default_rng(seed)
    
    # ---- initialize values to zero ----
    V = {s: 0.0 for s in env.all_states()}
    
    episode_rewards = []
    td_errors_history = []
    
    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0.0
        ep_td_errors = []
        done = False
        
        while not done:
            # ---- select action from policy ----
            action_probs = policy[state]
            action = local_rng.choice(env.n_actions, p=action_probs)
            
            # ---- take action ----
            next_state, reward, done = env.step(action)
            total_reward += reward
            
            # ---- compute TD error ----
            td_target = reward + gamma * V[next_state] * (1 - int(done))
            td_error = td_target - V[state]
            ep_td_errors.append(td_error)
            
            # ---- update value estimate ----
            V[state] = V[state] + alpha * td_error
            
            state = next_state
        
        episode_rewards.append(total_reward)
        td_errors_history.append(ep_td_errors)
    
    return V, episode_rewards, td_errors_history


print("td0_prediction() defined.")

---
## 8. TD(0) Prediction on GridWorld

In [ ]:
# ============================================================
#  Run TD(0) on GridWorld with uniform random policy
# ============================================================

gw = GridWorld()

# ---- uniform random policy (equal probability for all 4 actions) ----
uniform_policy = {
    s: np.array([0.25, 0.25, 0.25, 0.25])
    for s in gw.all_states()
}

# ---- run TD(0) for more episodes to ensure convergence ----
V_td, ep_rewards_td, td_errors_hist = td0_prediction(
    gw, uniform_policy, n_episodes=5000, alpha=ALPHA, gamma=GAMMA
)

print("Learned V(s) under uniform random policy:")
print("-" * 40)
for r in range(gw.rows):
    row_vals = [f"{V_td[(r, c)]:7.2f}" for c in range(gw.cols)]
    print("  ".join(row_vals))

# ---- true values for uniform random policy on 4x4 grid (undiscounted) ----
# These are the analytically known values from Sutton & Barto Example 4.1
V_true = {
    (0,0): -14, (0,1): -11, (0,2): -8, (0,3): -5,
    (1,0): -11, (1,1): -8,  (1,2): -5, (1,3): -2,
    (2,0): -8,  (2,1): -5,  (2,2): -2, (2,3): -1,
    (3,0): -5,  (3,1): -2,  (3,2): -1, (3,3): 0,
}

# Note: These approximate true values assume the random walk distances.
# TD(0) should converge close to these with enough episodes.
print("\nTrue V(s) for comparison:")
print("-" * 40)
for r in range(gw.rows):
    row_vals = [f"{V_true[(r, c)]:7.2f}" for c in range(gw.cols)]
    print("  ".join(row_vals))

---
## 9. TD(0) Value Convergence Visualization

In [ ]:
# ============================================================
#  TD(0) Value Convergence for Selected States
# ============================================================

# ---- track value evolution over episodes ----
tracked_states = [(0, 0), (0, 3), (2, 0), (1, 1)]
state_colors = [C_PRIMARY, C_SECONDARY, C_TERTIARY, C_ACCENT]

def td0_with_tracking(
    env, policy, n_episodes, alpha, gamma, tracked_states, seed=SEED
):
    """TD(0) prediction with value tracking for convergence plots.
    
    Args:
        env: GridWorld environment
        policy: state -> action probability mapping
        n_episodes: number of episodes
        alpha: learning rate
        gamma: discount factor
        tracked_states: list of states to track, shape (n_tracked,)
        seed: random seed
    
    Returns:
        V: final value estimates
        value_history: dict mapping state -> list of values, shape (n_episodes,)
    """
    local_rng = np.random.default_rng(seed)
    V = {s: 0.0 for s in env.all_states()}
    value_history = {s: [] for s in tracked_states}
    
    for ep in range(n_episodes):
        state = env.reset()
        done = False
        
        while not done:
            action = local_rng.choice(env.n_actions, p=policy[state])
            next_state, reward, done = env.step(action)
            td_target = reward + gamma * V[next_state] * (1 - int(done))
            V[state] += alpha * (td_target - V[state])
            state = next_state
        
        # ---- record values after each episode ----
        for s in tracked_states:
            value_history[s].append(V[s])
    
    return V, value_history


_, value_hist = td0_with_tracking(
    gw, uniform_policy, 5000, ALPHA, GAMMA, tracked_states
)

# ---- plot convergence curves ----
fig, ax = plt.subplots(figsize=(12, 5))

for s, color in zip(tracked_states, state_colors):
    ax.plot(value_hist[s], color=color, label=f'V{s}', alpha=0.8)
    ax.axhline(y=V_true[s], color=color, linestyle='--', alpha=0.5, linewidth=1)

ax.set_xlabel('Episode')
ax.set_ylabel('Estimated Value V(s)')
ax.set_title('TD(0) Value Convergence for Selected States')
ax.legend(loc='lower left', ncol=2)

plt.tight_layout()
plt.show()

print("Dashed lines show approximate true values.")

---
## 10. CliffWalking Environment

In [ ]:
# ============================================================
#  CliffWalking Environment (4x12 grid)
# ============================================================

class CliffWalkingEnv:
    """
    The Cliff Walking environment from Sutton & Barto Example 6.6.
    
    4x12 grid world:
    - Start: (3, 0) — bottom-left
    - Goal:  (3, 11) — bottom-right (terminal)
    - Cliff: (3, 1) through (3, 10) — stepping here gives -100 and resets to start
    - Regular step reward: -1
    
    Actions: 0=UP, 1=RIGHT, 2=DOWN, 3=LEFT
    States are encoded as integers: state = row * cols + col
    """
    
    UP, RIGHT, DOWN, LEFT = 0, 1, 2, 3
    ACTION_NAMES = ['UP', 'RIGHT', 'DOWN', 'LEFT']
    ACTION_DELTAS = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    ACTION_ARROWS = ['\u2191', '\u2192', '\u2193', '\u2190']  # arrows for visualization
    
    def __init__(self, rows: int = 4, cols: int = 12):
        """
        Args:
            rows: number of rows (default 4)
            cols: number of columns (default 12)
        """
        self.rows = rows
        self.cols = cols
        self.n_states = rows * cols
        self.n_actions = 4
        self.start = (3, 0)
        self.goal = (3, 11)
        # ---- cliff cells ----
        self.cliff = {(3, c) for c in range(1, 11)}
        self.state = None
    
    def state_to_idx(self, state: Tuple[int, int]) -> int:
        """Convert (row, col) to integer index."""
        return state[0] * self.cols + state[1]
    
    def idx_to_state(self, idx: int) -> Tuple[int, int]:
        """Convert integer index to (row, col)."""
        return (idx // self.cols, idx % self.cols)
    
    def reset(self) -> int:
        """Reset to start position.
        
        Returns:
            state: integer state index for the start position
        """
        self.state = self.start
        return self.state_to_idx(self.state)
    
    def is_cliff(self, state: Tuple[int, int]) -> bool:
        """Check if a state is on the cliff."""
        return state in self.cliff
    
    def is_terminal(self, state: Tuple[int, int]) -> bool:
        """Check if a state is the goal."""
        return state == self.goal
    
    def step(self, action: int) -> Tuple[int, float, bool]:
        """Take an action in the environment.
        
        Args:
            action: integer action (0=UP, 1=RIGHT, 2=DOWN, 3=LEFT)
        
        Returns:
            next_state_idx: integer index of next state
            reward: -1 for normal step, -100 for cliff
            done: True if agent reaches the goal
        """
        if self.is_terminal(self.state):
            return self.state_to_idx(self.state), 0.0, True
        
        dr, dc = self.ACTION_DELTAS[action]
        new_r = max(0, min(self.rows - 1, self.state[0] + dr))
        new_c = max(0, min(self.cols - 1, self.state[1] + dc))
        new_state = (new_r, new_c)
        
        if self.is_cliff(new_state):
            # ---- cliff: large penalty and reset to start ----
            self.state = self.start
            return self.state_to_idx(self.state), -100.0, False
        
        self.state = new_state
        done = self.is_terminal(self.state)
        return self.state_to_idx(self.state), -1.0, done


# Quick test
cliff_env = CliffWalkingEnv()
s = cliff_env.reset()
print(f"Start state index: {s}, position: {cliff_env.idx_to_state(s)}")
s, r, d = cliff_env.step(CliffWalkingEnv.RIGHT)
print(f"After RIGHT: state={s}, pos={cliff_env.idx_to_state(s)}, reward={r}, done={d}")
print(f"  (Fell off cliff! Reset to start.)")
s, r, d = cliff_env.step(CliffWalkingEnv.UP)
print(f"After UP:    state={s}, pos={cliff_env.idx_to_state(s)}, reward={r}, done={d}")

---
## 11. CliffWalking Grid Visualization

In [ ]:
# ============================================================
#  Visualize the CliffWalking Grid
# ============================================================

def render_cliff_grid(env: CliffWalkingEnv, title: str = "CliffWalking Environment"):
    """Render the CliffWalking grid showing start, goal, and cliff.
    
    Args:
        env: CliffWalkingEnv instance
        title: plot title
    """
    fig, ax = plt.subplots(figsize=(14, 4))
    
    for r in range(env.rows):
        for c in range(env.cols):
            pos = (r, c)
            if pos == env.start:
                color = C_PRIMARY
                label = 'S'
            elif pos == env.goal:
                color = C_TERTIARY
                label = 'G'
            elif env.is_cliff(pos):
                color = C_SECONDARY
                label = 'X'
            else:
                color = 'white'
                label = ''
            
            rect = mpatches.FancyBboxPatch(
                (c, env.rows - 1 - r), 1, 1,
                boxstyle="round,pad=0.02",
                facecolor=color, edgecolor='gray', alpha=0.6, linewidth=1.5
            )
            ax.add_patch(rect)
            if label:
                ax.text(c + 0.5, env.rows - 1 - r + 0.5, label,
                       ha='center', va='center', fontsize=14, fontweight='bold',
                       color='white' if color != 'white' else 'black')
    
    ax.set_xlim(0, env.cols)
    ax.set_ylim(0, env.rows)
    ax.set_aspect('equal')
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    ax.set_title(title)
    ax.set_xticks(range(env.cols))
    ax.set_yticks(range(env.rows))
    ax.set_xticklabels(range(env.cols))
    ax.set_yticklabels(range(env.rows - 1, -1, -1))
    ax.grid(False)
    
    # ---- legend ----
    legend_elements = [
        mpatches.Patch(facecolor=C_PRIMARY, alpha=0.6, label='Start (S)'),
        mpatches.Patch(facecolor=C_TERTIARY, alpha=0.6, label='Goal (G)'),
        mpatches.Patch(facecolor=C_SECONDARY, alpha=0.6, label='Cliff (X)'),
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.show()


render_cliff_grid(cliff_env)

---
## 12. Helper Functions: Epsilon-Greedy and Policy Extraction

In [ ]:
# ============================================================
#  Helper Functions
# ============================================================

def epsilon_greedy_action(
    Q: np.ndarray,
    state: int,
    epsilon: float,
    n_actions: int,
    local_rng: np.random.Generator
) -> int:
    """Select action using epsilon-greedy policy.
    
    Args:
        Q: action-value table, shape (n_states, n_actions)
        state: current state index
        epsilon: exploration probability
        n_actions: number of available actions
        local_rng: random number generator
    
    Returns:
        action: selected action index
    """
    if local_rng.random() < epsilon:
        return local_rng.integers(n_actions)
    else:
        # ---- break ties randomly ----
        max_q = np.max(Q[state])
        best_actions = np.where(Q[state] == max_q)[0]
        return local_rng.choice(best_actions)


def extract_policy(Q: np.ndarray, n_states: int, n_actions: int) -> np.ndarray:
    """Extract greedy policy from Q-values.
    
    Args:
        Q: action-value table, shape (n_states, n_actions)
        n_states: number of states
        n_actions: number of actions
    
    Returns:
        policy: greedy action for each state, shape (n_states,)
    """
    return np.argmax(Q, axis=1)


def extract_path(
    env: CliffWalkingEnv,
    policy: np.ndarray,
    max_steps: int = 100
) -> List[Tuple[int, int]]:
    """Follow the greedy policy to extract the path from start to goal.
    
    Args:
        env: CliffWalkingEnv instance
        policy: greedy action for each state, shape (n_states,)
        max_steps: maximum number of steps to prevent infinite loops
    
    Returns:
        path: list of (row, col) positions visited
    """
    state_idx = env.reset()
    path = [env.idx_to_state(state_idx)]
    
    for _ in range(max_steps):
        action = policy[state_idx]
        state_idx, reward, done = env.step(action)
        path.append(env.state)
        if done:
            break
    
    return path


print("Helper functions defined: epsilon_greedy_action(), extract_policy(), extract_path()")

---
## 13. SARSA: On-Policy TD Control

In [ ]:
# ============================================================
#  SARSA: On-Policy TD Control
# ============================================================

def sarsa(
    env: CliffWalkingEnv,
    n_episodes: int,
    alpha: float,
    gamma: float,
    epsilon: float,
    seed: int = SEED
) -> Tuple[np.ndarray, List[float], np.ndarray]:
    """
    SARSA: On-policy TD control algorithm.
    
    Updates Q(S,A) using the action A' actually taken in the next state,
    making it sensitive to the exploration policy.
    
    Args:
        env: CliffWalkingEnv instance
        n_episodes: number of training episodes
        alpha: learning rate
        gamma: discount factor
        epsilon: exploration probability for epsilon-greedy
        seed: random seed
    
    Returns:
        Q: learned action-value table, shape (n_states, n_actions)
        episode_rewards: total reward per episode, shape (n_episodes,)
        policy: greedy policy derived from Q, shape (n_states,)
    """
    local_rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.n_actions))
    episode_rewards = []
    
    for ep in range(n_episodes):
        state = env.reset()
        action = epsilon_greedy_action(Q, state, epsilon, env.n_actions, local_rng)
        total_reward = 0.0
        done = False
        
        while not done:
            # ---- take action, observe reward and next state ----
            next_state, reward, done = env.step(action)
            total_reward += reward
            
            # ---- choose next action from policy (on-policy) ----
            next_action = epsilon_greedy_action(
                Q, next_state, epsilon, env.n_actions, local_rng
            )
            
            # ---- SARSA update: use Q(S', A') ----
            td_target = reward + gamma * Q[next_state, next_action] * (1 - int(done))
            td_error = td_target - Q[state, action]
            Q[state, action] += alpha * td_error
            
            state = next_state
            action = next_action
        
        episode_rewards.append(total_reward)
    
    policy = extract_policy(Q, env.n_states, env.n_actions)
    return Q, episode_rewards, policy


print("sarsa() defined.")

---
## 14. Q-Learning: Off-Policy TD Control

In [ ]:
# ============================================================
#  Q-Learning: Off-Policy TD Control
# ============================================================

def q_learning(
    env: CliffWalkingEnv,
    n_episodes: int,
    alpha: float,
    gamma: float,
    epsilon: float,
    seed: int = SEED
) -> Tuple[np.ndarray, List[float], np.ndarray]:
    """
    Q-Learning: Off-policy TD control algorithm.
    
    Updates Q(S,A) using max_a' Q(S', a'), decoupling the behavior
    policy (epsilon-greedy) from the target policy (greedy).
    
    Args:
        env: CliffWalkingEnv instance
        n_episodes: number of training episodes
        alpha: learning rate
        gamma: discount factor
        epsilon: exploration probability for epsilon-greedy
        seed: random seed
    
    Returns:
        Q: learned action-value table, shape (n_states, n_actions)
        episode_rewards: total reward per episode, shape (n_episodes,)
        policy: greedy policy derived from Q, shape (n_states,)
    """
    local_rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.n_actions))
    episode_rewards = []
    
    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0.0
        done = False
        
        while not done:
            # ---- choose action using epsilon-greedy (behavior policy) ----
            action = epsilon_greedy_action(
                Q, state, epsilon, env.n_actions, local_rng
            )
            
            # ---- take action, observe reward and next state ----
            next_state, reward, done = env.step(action)
            total_reward += reward
            
            # ---- Q-Learning update: use max Q(S', a') ----
            td_target = reward + gamma * np.max(Q[next_state]) * (1 - int(done))
            td_error = td_target - Q[state, action]
            Q[state, action] += alpha * td_error
            
            state = next_state
        
        episode_rewards.append(total_reward)
    
    policy = extract_policy(Q, env.n_states, env.n_actions)
    return Q, episode_rewards, policy


print("q_learning() defined.")

---
## 15. Train SARSA and Q-Learning on CliffWalking

In [ ]:
# ============================================================
#  Train Both Algorithms on CliffWalking
# ============================================================

cliff_env = CliffWalkingEnv()

# ---- train SARSA ----
print("Training SARSA...")
Q_sarsa, rewards_sarsa, policy_sarsa = sarsa(
    cliff_env, N_EPISODES, ALPHA, GAMMA, EPSILON, seed=SEED
)
print(f"  Final 50-episode avg reward: {np.mean(rewards_sarsa[-50:]):.1f}")

# ---- train Q-Learning ----
print("\nTraining Q-Learning...")
Q_qlearn, rewards_qlearn, policy_qlearn = q_learning(
    cliff_env, N_EPISODES, ALPHA, GAMMA, EPSILON, seed=SEED
)
print(f"  Final 50-episode avg reward: {np.mean(rewards_qlearn[-50:]):.1f}")

# ---- extract paths ----
path_sarsa = extract_path(cliff_env, policy_sarsa)
path_qlearn = extract_path(cliff_env, policy_qlearn)

print(f"\nSARSA path length:      {len(path_sarsa)} steps")
print(f"Q-Learning path length: {len(path_qlearn)} steps")
print(f"\nSARSA path:      {path_sarsa}")
print(f"Q-Learning path: {path_qlearn}")

---
## 16. Episode Reward Curves: SARSA vs Q-Learning

In [ ]:
# ============================================================
#  SARSA vs Q-Learning: Episode Reward Comparison
# ============================================================

def running_average(data: List[float], window: int = 20) -> np.ndarray:
    """Compute a running average for smoothing.
    
    Args:
        data: raw data series, shape (n,)
        window: smoothing window size
    
    Returns:
        smoothed: running average, shape (n,)
    """
    cumsum = np.cumsum(np.insert(data, 0, 0))
    smoothed = np.zeros(len(data))
    for i in range(len(data)):
        start = max(0, i - window + 1)
        smoothed[i] = (cumsum[i + 1] - cumsum[start]) / (i - start + 1)
    return smoothed


# ---- smooth the reward curves ----
smooth_sarsa = running_average(rewards_sarsa, window=20)
smooth_qlearn = running_average(rewards_qlearn, window=20)

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(smooth_sarsa, color=C_PRIMARY, label='SARSA', alpha=0.9)
ax.plot(smooth_qlearn, color=C_SECONDARY, label='Q-Learning', alpha=0.9)

# ---- show raw rewards faintly ----
ax.plot(rewards_sarsa, color=C_PRIMARY, alpha=0.1, linewidth=0.5)
ax.plot(rewards_qlearn, color=C_SECONDARY, alpha=0.1, linewidth=0.5)

ax.set_xlabel('Episode')
ax.set_ylabel('Sum of Rewards')
ax.set_title('SARSA vs Q-Learning: Episode Rewards on CliffWalking')
ax.set_ylim(-200, 0)
ax.legend(loc='lower right', fontsize=13)
ax.axhline(y=-13, color='gray', linestyle=':', alpha=0.5, label='Optimal (-13)')

plt.tight_layout()
plt.show()

print("Key insight: SARSA achieves higher online rewards (safe path),")
print("while Q-Learning finds the optimal path but suffers more cliff falls during training.")

---
## 17. Q-Value Heatmaps

In [ ]:
# ============================================================
#  Q-Value Heatmaps for SARSA and Q-Learning
# ============================================================

def plot_q_heatmap(
    Q: np.ndarray,
    env: CliffWalkingEnv,
    title: str,
    ax: plt.Axes
):
    """Plot the max Q-value for each state as a heatmap.
    
    Args:
        Q: action-value table, shape (n_states, n_actions)
        env: CliffWalkingEnv instance
        title: subplot title
        ax: matplotlib axes to plot on
    """
    V = np.max(Q, axis=1).reshape(env.rows, env.cols)
    
    # ---- mask cliff cells for better visualization ----
    masked_V = np.copy(V)
    for (r, c) in env.cliff:
        masked_V[r, c] = np.nan
    
    im = ax.imshow(masked_V, cmap='RdYlGn', aspect='auto',
                   interpolation='nearest')
    plt.colorbar(im, ax=ax, shrink=0.8, label='max Q(s,a)')
    
    # ---- annotate cells ----
    for r in range(env.rows):
        for c in range(env.cols):
            if (r, c) in env.cliff:
                ax.text(c, r, 'X', ha='center', va='center',
                       fontsize=10, color='red', fontweight='bold')
            elif (r, c) == env.start:
                ax.text(c, r, f'S\n{V[r,c]:.0f}', ha='center', va='center',
                       fontsize=8, fontweight='bold')
            elif (r, c) == env.goal:
                ax.text(c, r, f'G\n{V[r,c]:.0f}', ha='center', va='center',
                       fontsize=8, fontweight='bold')
            else:
                ax.text(c, r, f'{V[r,c]:.0f}', ha='center', va='center',
                       fontsize=8)
    
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    ax.set_title(title)
    ax.set_xticks(range(env.cols))
    ax.set_yticks(range(env.rows))


fig, axes = plt.subplots(1, 2, figsize=(18, 4))

plot_q_heatmap(Q_sarsa, cliff_env, 'SARSA: max Q(s, a)', axes[0])
plot_q_heatmap(Q_qlearn, cliff_env, 'Q-Learning: max Q(s, a)', axes[1])

plt.tight_layout()
plt.show()

---
## 18. Path Comparison: SARSA vs Q-Learning

In [ ]:
# ============================================================
#  Path Comparison on the Grid
# ============================================================

def render_path_on_grid(
    env: CliffWalkingEnv,
    path: List[Tuple[int, int]],
    title: str,
    path_color: str,
    ax: plt.Axes
):
    """Draw a path on the CliffWalking grid.
    
    Args:
        env: CliffWalkingEnv instance
        path: list of (row, col) positions
        title: subplot title
        path_color: color for the path line
        ax: matplotlib axes
    """
    # ---- draw grid cells ----
    for r in range(env.rows):
        for c in range(env.cols):
            pos = (r, c)
            if pos == env.start:
                color = C_PRIMARY
            elif pos == env.goal:
                color = C_TERTIARY
            elif env.is_cliff(pos):
                color = C_SECONDARY
            else:
                color = '#f0f0f0'
            
            rect = mpatches.Rectangle(
                (c, env.rows - 1 - r), 1, 1,
                facecolor=color, edgecolor='gray', alpha=0.5, linewidth=1
            )
            ax.add_patch(rect)
    
    # ---- draw path ----
    if len(path) > 1:
        path_x = [c + 0.5 for (r, c) in path]
        path_y = [env.rows - 1 - r + 0.5 for (r, c) in path]
        ax.plot(path_x, path_y, color=path_color, linewidth=3, alpha=0.8,
               marker='o', markersize=5, zorder=5)
        # ---- mark start and end ----
        ax.plot(path_x[0], path_y[0], 'o', color=path_color, markersize=10, zorder=6)
        ax.plot(path_x[-1], path_y[-1], 's', color=path_color, markersize=10, zorder=6)
    
    ax.set_xlim(0, env.cols)
    ax.set_ylim(0, env.rows)
    ax.set_aspect('equal')
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    ax.set_title(f'{title} ({len(path)-1} steps)')
    ax.set_xticks(range(env.cols))
    ax.set_yticks(range(env.rows))
    ax.set_xticklabels(range(env.cols))
    ax.set_yticklabels(range(env.rows - 1, -1, -1))
    ax.grid(False)


fig, axes = plt.subplots(1, 2, figsize=(18, 4))

render_path_on_grid(cliff_env, path_sarsa, 'SARSA Path (Safe)', C_PRIMARY, axes[0])
render_path_on_grid(cliff_env, path_qlearn, 'Q-Learning Path (Optimal)', C_SECONDARY, axes[1])

plt.tight_layout()
plt.show()

print("SARSA learns the safe path (away from cliff) because it accounts")
print("for exploratory actions that might cause cliff falls.")
print("\nQ-Learning learns the optimal path (along cliff edge) because")
print("it evaluates the greedy policy, ignoring exploration risk.")

---
## 19. Policy Arrow Visualization

In [ ]:
# ============================================================
#  Policy Arrow Visualization
# ============================================================

def render_policy_arrows(
    env: CliffWalkingEnv,
    policy: np.ndarray,
    title: str,
    ax: plt.Axes
):
    """Render the greedy policy as arrows on the grid.
    
    Args:
        env: CliffWalkingEnv instance
        policy: greedy action for each state, shape (n_states,)
        title: subplot title
        ax: matplotlib axes
    """
    arrows = CliffWalkingEnv.ACTION_ARROWS
    
    for r in range(env.rows):
        for c in range(env.cols):
            pos = (r, c)
            if pos == env.start:
                color = C_PRIMARY
            elif pos == env.goal:
                color = C_TERTIARY
            elif env.is_cliff(pos):
                color = C_SECONDARY
            else:
                color = '#f0f0f0'
            
            rect = mpatches.Rectangle(
                (c, env.rows - 1 - r), 1, 1,
                facecolor=color, edgecolor='gray', alpha=0.4, linewidth=1
            )
            ax.add_patch(rect)
            
            state_idx = env.state_to_idx(pos)
            if not env.is_cliff(pos) and not env.is_terminal(pos):
                ax.text(c + 0.5, env.rows - 1 - r + 0.5,
                       arrows[policy[state_idx]],
                       ha='center', va='center', fontsize=14)
            elif env.is_terminal(pos):
                ax.text(c + 0.5, env.rows - 1 - r + 0.5, 'G',
                       ha='center', va='center', fontsize=14, fontweight='bold')
            elif env.is_cliff(pos):
                ax.text(c + 0.5, env.rows - 1 - r + 0.5, 'X',
                       ha='center', va='center', fontsize=12, color='red')
    
    ax.set_xlim(0, env.cols)
    ax.set_ylim(0, env.rows)
    ax.set_aspect('equal')
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    ax.set_title(title)
    ax.set_xticks(range(env.cols))
    ax.set_yticks(range(env.rows))
    ax.set_xticklabels(range(env.cols))
    ax.set_yticklabels(range(env.rows - 1, -1, -1))
    ax.grid(False)


fig, axes = plt.subplots(1, 2, figsize=(18, 4))

render_policy_arrows(cliff_env, policy_sarsa, 'SARSA Policy', axes[0])
render_policy_arrows(cliff_env, policy_qlearn, 'Q-Learning Policy', axes[1])

plt.tight_layout()
plt.show()

---
## 20. TD Error Distribution

In [ ]:
# ============================================================
#  TD Error Distribution from TD(0) Prediction
# ============================================================

# ---- collect all TD errors from the prediction run ----
all_td_errors = [err for ep_errors in td_errors_hist for err in ep_errors]

# ---- split into early and late episodes ----
n_ep = len(td_errors_hist)
early_errors = [err for ep_errors in td_errors_hist[:n_ep//5] for err in ep_errors]
late_errors = [err for ep_errors in td_errors_hist[-n_ep//5:] for err in ep_errors]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- overall distribution ----
axes[0].hist(all_td_errors, bins=60, color=C_PRIMARY, alpha=0.7, edgecolor='white')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
axes[0].set_xlabel('TD Error')
axes[0].set_ylabel('Frequency')
axes[0].set_title('TD Error Distribution (All Episodes)')
axes[0].text(0.02, 0.95, f'Mean: {np.mean(all_td_errors):.3f}\nStd: {np.std(all_td_errors):.3f}',
            transform=axes[0].transAxes, va='top', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# ---- early vs late comparison ----
axes[1].hist(early_errors, bins=40, color=C_SECONDARY, alpha=0.6,
            label=f'Early (first {n_ep//5} eps)', edgecolor='white')
axes[1].hist(late_errors, bins=40, color=C_TERTIARY, alpha=0.6,
            label=f'Late (last {n_ep//5} eps)', edgecolor='white')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
axes[1].set_xlabel('TD Error')
axes[1].set_ylabel('Frequency')
axes[1].set_title('TD Errors: Early vs Late Training')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f"Early episodes — mean TD error: {np.mean(early_errors):.4f}, std: {np.std(early_errors):.4f}")
print(f"Late episodes  — mean TD error: {np.mean(late_errors):.4f}, std: {np.std(late_errors):.4f}")
print("\nAs training progresses, TD errors shrink toward zero — values are converging.")

---
## 21. Multiple Runs: Statistical Comparison

In [ ]:
# ============================================================
#  Multiple Independent Runs for Robust Comparison
# ============================================================

N_RUNS = 30
sarsa_all_rewards = np.zeros((N_RUNS, N_EPISODES))
qlearn_all_rewards = np.zeros((N_RUNS, N_EPISODES))

for run in range(N_RUNS):
    seed_run = SEED + run
    env_run = CliffWalkingEnv()
    
    _, sarsa_all_rewards[run], _ = sarsa(
        env_run, N_EPISODES, ALPHA, GAMMA, EPSILON, seed=seed_run
    )
    _, qlearn_all_rewards[run], _ = q_learning(
        env_run, N_EPISODES, ALPHA, GAMMA, EPSILON, seed=seed_run
    )

# ---- compute mean and std across runs ----
sarsa_mean = np.mean(sarsa_all_rewards, axis=0)
sarsa_std = np.std(sarsa_all_rewards, axis=0)
qlearn_mean = np.mean(qlearn_all_rewards, axis=0)
qlearn_std = np.std(qlearn_all_rewards, axis=0)

# ---- smooth for readability ----
window = 20
sarsa_smooth = running_average(sarsa_mean.tolist(), window)
qlearn_smooth = running_average(qlearn_mean.tolist(), window)
sarsa_std_smooth = running_average(sarsa_std.tolist(), window)
qlearn_std_smooth = running_average(qlearn_std.tolist(), window)

episodes = np.arange(N_EPISODES)

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(episodes, sarsa_smooth, color=C_PRIMARY, label='SARSA')
ax.fill_between(episodes,
                sarsa_smooth - sarsa_std_smooth,
                sarsa_smooth + sarsa_std_smooth,
                alpha=0.15, color=C_PRIMARY)

ax.plot(episodes, qlearn_smooth, color=C_SECONDARY, label='Q-Learning')
ax.fill_between(episodes,
                qlearn_smooth - qlearn_std_smooth,
                qlearn_smooth + qlearn_std_smooth,
                alpha=0.15, color=C_SECONDARY)

ax.axhline(y=-13, color='gray', linestyle=':', alpha=0.5, linewidth=1)
ax.text(N_EPISODES - 50, -11, 'Optimal (-13)', fontsize=10, color='gray')

ax.set_xlabel('Episode')
ax.set_ylabel('Sum of Rewards (averaged over runs)')
ax.set_title(f'SARSA vs Q-Learning: {N_RUNS} Independent Runs')
ax.set_ylim(-150, 0)
ax.legend(loc='lower right', fontsize=13)

plt.tight_layout()
plt.show()

print(f"SARSA     — final avg reward: {sarsa_mean[-50:].mean():.1f} +/- {sarsa_std[-50:].mean():.1f}")
print(f"Q-Learning — final avg reward: {qlearn_mean[-50:].mean():.1f} +/- {qlearn_std[-50:].mean():.1f}")

---
## 22. Learning Rate Sensitivity Analysis

In [ ]:
# ============================================================
#  Learning Rate Sensitivity
# ============================================================

alphas = [0.01, 0.05, 0.1, 0.3, 0.5]
alpha_colors = [C_PRIMARY, C_SECONDARY, C_TERTIARY, C_ACCENT, C_HIGHLIGHT]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for alpha_val, color in zip(alphas, alpha_colors):
    env_lr = CliffWalkingEnv()
    
    # ---- SARSA ----
    _, rewards_s, _ = sarsa(env_lr, N_EPISODES, alpha_val, GAMMA, EPSILON, seed=SEED)
    smooth_s = running_average(rewards_s, window=30)
    axes[0].plot(smooth_s, color=color, label=f'alpha={alpha_val}', alpha=0.8)
    
    # ---- Q-Learning ----
    _, rewards_q, _ = q_learning(env_lr, N_EPISODES, alpha_val, GAMMA, EPSILON, seed=SEED)
    smooth_q = running_average(rewards_q, window=30)
    axes[1].plot(smooth_q, color=color, label=f'alpha={alpha_val}', alpha=0.8)

for i, title in enumerate(['SARSA', 'Q-Learning']):
    axes[i].set_xlabel('Episode')
    axes[i].set_ylabel('Sum of Rewards')
    axes[i].set_title(f'{title}: Learning Rate Sensitivity')
    axes[i].set_ylim(-200, 0)
    axes[i].legend(fontsize=10, loc='lower right')
    axes[i].axhline(y=-13, color='gray', linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()

print("Higher learning rates converge faster but may be less stable.")
print("Very low rates (0.01) converge too slowly within 500 episodes.")

---
## 23. Exploration Rate (Epsilon) Analysis

In [ ]:
# ============================================================
#  Exploration Rate (Epsilon) Analysis
# ============================================================

epsilons = [0.01, 0.05, 0.1, 0.2, 0.4]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for eps_val, color in zip(epsilons, alpha_colors):
    env_eps = CliffWalkingEnv()
    
    # ---- SARSA ----
    _, rewards_s, _ = sarsa(env_eps, N_EPISODES, ALPHA, GAMMA, eps_val, seed=SEED)
    smooth_s = running_average(rewards_s, window=30)
    axes[0].plot(smooth_s, color=color, label=f'eps={eps_val}', alpha=0.8)
    
    # ---- Q-Learning ----
    _, rewards_q, _ = q_learning(env_eps, N_EPISODES, ALPHA, GAMMA, eps_val, seed=SEED)
    smooth_q = running_average(rewards_q, window=30)
    axes[1].plot(smooth_q, color=color, label=f'eps={eps_val}', alpha=0.8)

for i, title in enumerate(['SARSA', 'Q-Learning']):
    axes[i].set_xlabel('Episode')
    axes[i].set_ylabel('Sum of Rewards')
    axes[i].set_title(f'{title}: Epsilon Sensitivity')
    axes[i].set_ylim(-200, 0)
    axes[i].legend(fontsize=10, loc='lower right')
    axes[i].axhline(y=-13, color='gray', linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()

print("SARSA is more sensitive to epsilon — higher exploration causes more cliff falls,")
print("pushing it further from the cliff edge.")
print("Q-Learning's greedy target policy is unaffected by epsilon in the update rule.")

---
## 24. TD(0) Convergence Verification

In [ ]:
# ============================================================
#  Verification: TD(0) Convergence
# ============================================================

# ---- run TD(0) with many episodes for reliable convergence ----
gw_verify = GridWorld()
V_verify, _, _ = td0_prediction(
    gw_verify, uniform_policy, n_episodes=50000, alpha=0.01, gamma=GAMMA, seed=SEED
)

# ---- check convergence to approximate true values ----
max_error = 0.0
print("TD(0) Convergence Check:")
print(f"{'State':<12} {'Learned':>10} {'True':>10} {'Error':>10}")
print("-" * 45)

for r in range(gw_verify.rows):
    for c in range(gw_verify.cols):
        s = (r, c)
        learned = V_verify[s]
        true_val = V_true[s]
        error = abs(learned - true_val)
        max_error = max(max_error, error)
        print(f"{str(s):<12} {learned:>10.2f} {true_val:>10.2f} {error:>10.3f}")

# ---- PASS/FAIL check ----
tolerance = 2.0  # allow some tolerance due to stochastic nature
status = "PASS" if max_error < tolerance else "FAIL"
print(f"\nMax absolute error: {max_error:.3f}")
print(f"TD(0) converges to approximate true values (tol={tolerance}): [{status}]")

---
## 25. SARSA Path Verification

In [ ]:
# ============================================================
#  Verification: SARSA Finds the Safe Path
# ============================================================

print("SARSA Path Analysis")
print("=" * 50)
print(f"Path: {path_sarsa}")
print(f"Path length: {len(path_sarsa) - 1} steps")

# ---- check that path reaches the goal ----
sarsa_reaches_goal = path_sarsa[-1] == cliff_env.goal
status = "PASS" if sarsa_reaches_goal else "FAIL"
print(f"\nSARSA reaches goal: [{status}]")

# ---- check that path avoids cliff-adjacent cells (except start and goal) ----
# The "safe" path goes UP from start, across, then DOWN to goal
# Middle portion of path should not be in row 3 (cliff row)
cliff_adjacent = False
for pos in path_sarsa[1:-1]:  # exclude start and goal
    if pos[0] == 3:  # row 3 is the cliff row
        cliff_adjacent = True
        break

status = "PASS" if not cliff_adjacent else "FAIL"
print(f"SARSA avoids cliff row in middle of path: [{status}]")

# ---- check path goes through upper rows ----
min_row_middle = min(pos[0] for pos in path_sarsa[1:-1]) if len(path_sarsa) > 2 else 3
status = "PASS" if min_row_middle < 3 else "FAIL"
print(f"SARSA path goes through upper rows (min row = {min_row_middle}): [{status}]")

---
## 26. Q-Learning Path Verification

In [ ]:
# ============================================================
#  Verification: Q-Learning Finds the Optimal Path
# ============================================================

print("Q-Learning Path Analysis")
print("=" * 50)
print(f"Path: {path_qlearn}")
print(f"Path length: {len(path_qlearn) - 1} steps")

# ---- check that path reaches the goal ----
qlearn_reaches_goal = path_qlearn[-1] == cliff_env.goal
status = "PASS" if qlearn_reaches_goal else "FAIL"
print(f"\nQ-Learning reaches goal: [{status}]")

# ---- check that Q-Learning finds the optimal path (shortest = 13 steps) ----
# Optimal path goes along row 3 (cliff edge) or any shortest path = 13 steps
optimal_length = 13  # start(3,0) -> up -> across row 2 -> down to goal OR along bottom
qlearn_length = len(path_qlearn) - 1
status = "PASS" if qlearn_length == optimal_length else "FAIL"
print(f"Q-Learning path is optimal length ({qlearn_length} steps, expected {optimal_length}): [{status}]")

# ---- check that Q-Learning has lower online reward than SARSA ----
sarsa_avg = np.mean(rewards_sarsa)
qlearn_avg = np.mean(rewards_qlearn)
status = "PASS" if qlearn_avg < sarsa_avg else "FAIL"
print(f"\nQ-Learning online reward ({qlearn_avg:.1f}) < SARSA online reward ({sarsa_avg:.1f}): [{status}]")
print("  (Q-Learning suffers more cliff falls during epsilon-greedy exploration)")

---
## 27. Summary Verification

In [ ]:
# ============================================================
#  Summary Verification
# ============================================================

print("=" * 60)
print("  TEMPORAL DIFFERENCE LEARNING — VERIFICATION SUMMARY")
print("=" * 60)

# ---- 1. TD(0) convergence ----
td0_pass = max_error < tolerance
status = "PASS" if td0_pass else "FAIL"
print(f"\n  1. TD(0) converges to true values ........... [{status}]")

# ---- 2. SARSA safe path ----
sarsa_safe = not cliff_adjacent and sarsa_reaches_goal
status = "PASS" if sarsa_safe else "FAIL"
print(f"  2. SARSA finds safe path (avoids cliff) ..... [{status}]")

# ---- 3. Q-Learning optimal path ----
qlearn_optimal = qlearn_reaches_goal and qlearn_length == optimal_length
status = "PASS" if qlearn_optimal else "FAIL"
print(f"  3. Q-Learning finds optimal path ............ [{status}]")

# ---- 4. Q-Learning lower online reward ----
qlearn_lower = qlearn_avg < sarsa_avg
status = "PASS" if qlearn_lower else "FAIL"
print(f"  4. Q-Learning has lower online reward ........ [{status}]")

# ---- 5. Both reach goal ----
both_reach = sarsa_reaches_goal and qlearn_reaches_goal
status = "PASS" if both_reach else "FAIL"
print(f"  5. Both policies reach the goal .............. [{status}]")

# ---- overall ----
all_pass = td0_pass and sarsa_safe and qlearn_optimal and qlearn_lower and both_reach
overall = "ALL PASS" if all_pass else "SOME FAILED"
print(f"\n{'=' * 60}")
print(f"  Overall: [{overall}]")
print(f"{'=' * 60}")

---
## 28. Key Takeaways

### What We Demonstrated

1. **TD(0) Prediction** bootstraps value estimates using single-step transitions, converging to the true value function with lower variance than Monte Carlo methods.

2. **SARSA** (on-policy) learns Q-values that reflect the actual behavior policy, including exploration. On CliffWalking, this leads to the **safe path** that avoids the cliff edge, because SARSA accounts for the risk of epsilon-greedy exploration near the cliff.

3. **Q-Learning** (off-policy) learns Q-values for the optimal greedy policy regardless of exploration. It finds the **optimal (shortest) path** along the cliff edge, but suffers worse online performance due to cliff falls during training.

4. The **on-policy vs off-policy** distinction has real practical consequences: SARSA optimizes expected performance *under the current policy* (including exploration), while Q-Learning optimizes the best possible performance *if exploration were turned off*.

### Next Steps

- **Expected SARSA**: a middle ground that uses the expected Q-value under the policy instead of a sampled action
- **n-step TD**: interpolate between TD(0) and Monte Carlo with multi-step returns
- **Eligibility traces**: TD(lambda) for smooth bias-variance control
- **Function approximation**: scale TD methods to large/continuous state spaces